# 01 — QuPath ingest and geometry QC

This notebook is an output-first view of the production `ingest` and `geometry` stages. It contains no ingestion, rasterization, duplicate-selection, or QC algorithms. The immutable QuPath cohort manifest supplies donor/image identity, panel mappings, geometry, pixel calibration, and source fingerprints; production APIs own all writes.

Run the status and inspection cells first. Set `RUN_STAGES = True` only when you intend to create the content-addressed stage outputs.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from phenocycler.artifacts import StageManifest
from phenocycler.config import load_config
from phenocycler.expression import read_single_partition
from phenocycler.pipeline import RunContext, resolve_run_context, run_stage, status

CONFIG_PATH = None  # optionally set a repository-relative Path to another config.ini
cfg = load_config(CONFIG_PATH)
context: RunContext = resolve_run_context(cfg)
print(f"run_id={context.run_id}  donors={len(context.donors)}  root={context.run_root}")

In [ ]:
# Evidence-backed and non-mutating: validates manifests, exact donor sets, config, code and cached file stats.
status_code = status(context)
print(f"status return code: {status_code}")

## Source contract

There is exactly one manifest record per donor/image. Images sharing a panel ID must share an exact marker-to-channel mapping. Fast validation does not reread unchanged qptiffs; use content validation outside routine status when a full audit is required.

In [ ]:
cohort_rows = [
    {
        "donor_id": image.donor_id,
        "image_id": image.image_id,
        "panel_id": image.panel_id,
        "channels": len(image.channel_map),
        "pixel_size_um_x": image.pixel_size_um_x,
        "pixel_size_um_y": image.pixel_size_um_y,
        "segmentation_version": image.segmentation_version,
        "objects": image.cell_geometry.feature_count,
        "geometry_content": image.content_id[:12],
    }
    for image in context.cohort.images
]
display(pd.DataFrame(cohort_rows))
display(pd.DataFrame([
    {"panel_id": panel.panel_id, "channels": len(panel.channels), "content_id": panel.content_id[:12]}
    for panel in context.cohort.panels
]))

## Optional production execution

`run_stage` either validates an existing immutable stage or runs the production implementation and writes its content-derived manifest. It refuses partial output directories.

In [ ]:
RUN_STAGES = False

if RUN_STAGES:
    for stage_name in ("ingest", "geometry"):
        run_stage(context, stage_name)
else:
    print("Inspection only. Set RUN_STAGES=True to run ingest and geometry through production APIs.")

In [ ]:
manifest_rows = []
for stage_name in ("ingest", "geometry"):
    path = context.stage_manifest_path(stage_name)
    if path.exists():
        manifest = StageManifest.read_json(path)
        manifest_rows.append({
            "stage": stage_name,
            "method_version": manifest.method_version,
            "donors": len(manifest.completed_donors),
            "rows": manifest.output.total_rows,
            "schema": manifest.output.schema_sha256[:12],
            "objects": manifest.output.object_id_sha256[:12],
            "content": manifest.content_id[:12],
        })
display(pd.DataFrame(manifest_rows))

## Inspect one donor

Geometry QC preserves the complete ingest table and emits three separate decisions: `analysis_eligible`, `estimation_eligible`, and `spillover_context_eligible`. No marker intensity or REDSEA result participates in these flags.

In [ ]:
DONOR = context.donors[0]
required = (context.stage_manifest_path("ingest"), context.stage_manifest_path("geometry"))
if all(path.exists() for path in required):
    cells = read_single_partition(context.config.cells_dir, DONOR)
    geometry = read_single_partition(context.config.geometry_qc_dir, DONOR)
    print(f"donor {DONOR}: cells={len(cells):,}, geometry decisions={len(geometry):,}")
    eligibility = [
        "analysis_eligible", "estimation_eligible", "spillover_context_eligible"
    ]
    display(geometry[eligibility].mean().rename("eligible_fraction").to_frame())
    display(
        geometry.groupby("analysis_reason", dropna=False).size()
        .rename("cells").sort_values(ascending=False).to_frame().head(15)
    )
    preview_columns = [
        column for column in (
            "object_id", "analysis_eligible", "estimation_eligible",
            "spillover_context_eligible", "analysis_reasons",
            "raster_area_ratio", "nucleus_cell_area_ratio"
        ) if column in geometry
    ]
    display(geometry.loc[:, preview_columns].head())
else:
    print("Ingest/geometry artifacts are not complete yet; inspect status or run the optional stage cell.")

## Handoff

Proceed to notebook 02 only when both manifests are `CURRENT`. REDSEA consumes the immutable ingest and geometry universes; it no longer serves as a prerequisite for geometry QC.